# Notebook 04 — Optimización de Hiperparámetros (v3 — cuenca)

Busca los mejores hiperparámetros para los 3 modelos no-baseline usando
`ParameterSampler` de sklearn con `evaluar_con_panel_cv` como estrategia de CV.

Cada búsqueda se registra en MLflow con runs anidados:
```
run_padre: tuning_random_forest
  └── iter_000: n_estimators=120, max_depth=5 → AUC=0.991
  └── iter_001: n_estimators=80,  max_depth=9 → AUC=0.993
  └── ...
```

Los mejores hiperparámetros se guardan en `data/processed/best_params.json`
para ser consumidos por los Prefect flows del Bloque 4.

**Features:** 13 (CHIRPS × 5 + HydroSHEDS × 4 incluyendo ORDER + estacionalidad × 4)


## 0. Entorno y configuración

In [1]:
import sys
from pathlib import Path

_cwd = Path().resolve()
for _p in [_cwd, *_cwd.parents]:
    if (_p / "pyproject.toml").exists():
        _root = _p
        break
else:
    _root = _cwd.parent

sys.path.insert(0, str(_root / "src"))

import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.stats import randint, uniform
from sklearn.base import clone
from sklearn.model_selection import ParameterSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from pulearn import BaggingPuClassifier

import mlflow
import mlflow.sklearn

from experiment.config import load_config
from experiment.evaluate import panel_time_splits, evaluar_con_panel_cv
from experiment.train import make_pipeline

cfg = load_config(project_root=_root)
print(f"Config cargada — experimento: {cfg.mlflow.experiment_name}")
print(f"Features totales (all_features): {len(cfg.all_features)}")
print(f"  {cfg.all_features}")


Config cargada — experimento: antioquia_deslizamientos_semanal
Features totales (all_features): 14
  ['precip_acum_14d', 'precip_max_diario_14d', 'precip_dias_lluvia_14d', 'precip_acum_7d', 'precip_acum_3d', 'SUB_AREA', 'UP_AREA', 'DIST_MAIN', 'ORDER', 'semana_sin', 'semana_cos', 'mes_sin', 'mes_cos', 'soil_moisture_14d']


## 1. Configurar MLflow

In [2]:
mlflow.set_tracking_uri(cfg.mlflow_tracking_uri)
experiment = mlflow.set_experiment(cfg.mlflow.experiment_name)

print(f"MLflow URI        : {cfg.mlflow_tracking_uri}")
print(f"Experimento ID    : {experiment.experiment_id}")


MLflow URI        : sqlite:///C:\Users\Mateo Atehortua\Documents\riesgo-deslizamientos-mlops\mlruns\mlflow.db
Experimento ID    : 2


## 2. Cargar grid semana × cuenca

In [3]:
GRID_PATH = _root / "data" / "processed" / "grid_cuencas_v3.parquet"

df_model = pd.read_parquet(GRID_PATH)
# Restaurar tipo Period (se guardó como string)
df_model["anio_semana"] = pd.PeriodIndex(df_model["anio_semana"], freq="W")
df_model = df_model.sort_values("anio_semana").reset_index(drop=True)

print(f"Grid cargado: {df_model.shape}")
print(f"Columnas: {list(df_model.columns)}")


Grid cargado: (7629, 17)
Columnas: ['anio_semana', 'HYBAS_ID', 'n_eventos', 'deslizamiento', 'SUB_AREA', 'UP_AREA', 'DIST_MAIN', 'ORDER', 'semana_sin', 'semana_cos', 'mes_sin', 'mes_cos', 'precip_acum_14d', 'precip_max_diario_14d', 'precip_dias_lluvia_14d', 'precip_acum_7d', 'precip_acum_3d']


## 3. Definir features y target

Usamos `cfg.all_features` como fuente única de verdad (definido en el YAML).
Incluye ORDER (orden de Strahler) como feature HydroSHEDS adicional.


In [4]:
# Features disponibles en el parquet que coincidan con cfg.all_features
FEATURES = [f for f in cfg.all_features if f in df_model.columns]
TARGET   = cfg.target.nombre

X = df_model[FEATURES]
y = df_model[TARGET]

print(f"Features ({len(FEATURES)}): {FEATURES}")
print(f"Clase 1 (deslizamiento): {y.sum():,} ({y.mean():.2%})")
print(f"Clase 0 (sin evento)   : {(y==0).sum():,}")
print(f"NaN en X               : {X.isnull().sum().sum()}")


Features (13): ['precip_acum_14d', 'precip_max_diario_14d', 'precip_dias_lluvia_14d', 'precip_acum_7d', 'precip_acum_3d', 'SUB_AREA', 'UP_AREA', 'DIST_MAIN', 'ORDER', 'semana_sin', 'semana_cos', 'mes_sin', 'mes_cos']
Clase 1 (deslizamiento): 462 (6.06%)
Clase 0 (sin evento)   : 7,167
NaN en X               : 0


## 4. Verificación del CV de panel

In [5]:
print("Diagnóstico CV de panel:")
for fold, (tr, va) in enumerate(panel_time_splits(df_model, n_splits=4), 1):
    y_tr = y.loc[tr]
    y_va = y.loc[va]
    sem_tr = df_model.loc[tr, "anio_semana"]
    sem_va = df_model.loc[va, "anio_semana"]
    print(
        f"  Fold {fold}: train {sem_tr.min()}→{sem_tr.max()} (pos={y_tr.sum()}) | "
        f"val {sem_va.min()}→{sem_va.max()} (pos={y_va.sum()})"
    )


Diagnóstico CV de panel:
  Fold 1: train 2019-01-14/2019-01-20→2020-02-17/2020-02-23 (pos=24) | val 2020-02-24/2020-03-01→2020-11-16/2020-11-22 (pos=95)
  Fold 2: train 2019-01-14/2019-01-20→2020-11-16/2020-11-22 (pos=119) | val 2020-11-23/2020-11-29→2021-07-05/2021-07-11 (pos=99)
  Fold 3: train 2019-01-14/2019-01-20→2021-07-05/2021-07-11 (pos=218) | val 2021-07-19/2021-07-25→2022-04-18/2022-04-24 (pos=110)
  Fold 4: train 2019-01-14/2019-01-20→2022-04-18/2022-04-24 (pos=328) | val 2022-04-25/2022-05-01→2022-12-26/2023-01-01 (pos=134)


## 5. Espacios de búsqueda e hiperparámetros

`ParameterSampler` muestrea combinaciones de forma pseudo-aleatoria.
Los parámetros usan el prefijo `clf__` para dirigirse al paso del Pipeline.


In [6]:
N_ITER    = 20   # iteraciones por modelo — aumentar para búsqueda más exhaustiva
N_SPLITS  = 4
SEED      = 42
N_POS     = int(y.sum())   # BaggingPU necesita max_samples

ESPACIOS = {
    "logistic_regression": {
        "clf__C":      uniform(0.01, 10),           # C en [0.01, 10.01]
        "clf__solver": ["lbfgs", "liblinear"],
    },
    "random_forest": {
        "clf__n_estimators":    randint(50, 300),   # [50, 299]
        "clf__max_depth":       randint(3, 12),     # [3, 11]
        "clf__min_samples_leaf":randint(1, 20),     # [1, 19]
    },
    "bagging_pu": {
        "clf__n_estimators":            randint(10, 30),   # [10, 29]
        "clf__estimator__max_depth":    randint(3, 8),     # [3, 7]
    },
}

# Clasificadores base — sus params se sobreescribirán en cada iteración
CLASIFICADORES_BASE = {
    "logistic_regression": LogisticRegression(
        class_weight=cfg.target.class_weight,
        max_iter=500,
        random_state=SEED,
    ),
    "random_forest": RandomForestClassifier(
        class_weight=cfg.target.class_weight,
        random_state=SEED,
        n_jobs=-1,
    ),
    "bagging_pu": BaggingPuClassifier(
        estimator=RandomForestClassifier(random_state=SEED),
        max_samples=N_POS,
        n_jobs=-1,
        random_state=SEED,
    ),
}

print(f"Modelos a tunear : {list(ESPACIOS.keys())}")
print(f"Iteraciones/modelo: {N_ITER}")
print(f"Folds CV          : {N_SPLITS}")
print(f"Total evaluaciones: {len(ESPACIOS) * N_ITER * N_SPLITS}")


Modelos a tunear : ['logistic_regression', 'random_forest', 'bagging_pu']
Iteraciones/modelo: 20
Folds CV          : 4
Total evaluaciones: 240


## 6. Loop de tuning con MLflow nested runs

Cada modelo genera un run padre con N_ITER runs hijos.
La jerarquía es visible en la UI de MLflow (`mlflow ui`).


In [7]:
resultados_tuning = {}

tags_base = {
    **cfg.mlflow.run_tags,
    "fase":           "hyperparameter_tuning",
    "cv_estrategia":  "panel_time_split",
    "n_iter":         str(N_ITER),
    "granularidad":   "cuenca",
}

for nombre, espacio in ESPACIOS.items():
    print(f"\n{'='*55}")
    print(f"Tuning: {nombre}  ({N_ITER} iteraciones × {N_SPLITS} folds)")
    print(f"{'='*55}")

    clf_base = CLASIFICADORES_BASE[nombre]
    sampler  = list(ParameterSampler(espacio, n_iter=N_ITER, random_state=SEED))

    mejor_auc    = -1.0
    mejor_params = {}
    mejor_metricas = {}

    with mlflow.start_run(
        run_name=f"tuning_{nombre}",
        tags={**tags_base, "modelo": nombre},
    ) as parent_run:
        mlflow.log_params({
            "modelo":       nombre,
            "n_iter":       N_ITER,
            "cv_splits":    N_SPLITS,
            "n_features":   len(FEATURES),
            "n_instancias": len(X),
            "n_positivos":  N_POS,
        })

        for i, params in enumerate(sampler):
            with mlflow.start_run(run_name=f"iter_{i:03d}", nested=True):
                # Clonar clasificador base y aplicar params al pipeline completo
                pipeline = make_pipeline(clone(clf_base))
                pipeline.set_params(**params)

                metricas = evaluar_con_panel_cv(
                    pipeline, X, y, df_model, n_splits=N_SPLITS
                )

                # Serializar params (numpy int/float → Python nativo)
                params_log = {
                    k: (v.item() if hasattr(v, "item") else v)
                    for k, v in params.items()
                }
                mlflow.log_params(params_log)
                mlflow.log_metrics(metricas)

            auc = metricas["auc_roc_mean"]
            marker = " ◀ mejor" if auc > mejor_auc else ""
            print(
                f"  iter {i:03d}  AUC={auc:.4f}  "
                f"Recall={metricas['recall_mean']:.4f}  "
                f"params={params_log}{marker}"
            )

            if auc > mejor_auc:
                mejor_auc      = auc
                mejor_params   = params_log
                mejor_metricas = metricas

        # Loguear mejores en el run padre
        mlflow.log_params({f"best__{k}": v for k, v in mejor_params.items()})
        mlflow.log_metrics({f"best__{k}": v for k, v in mejor_metricas.items()})

    resultados_tuning[nombre] = {
        "best_params":   mejor_params,
        "best_metrics":  mejor_metricas,
    }
    print(f"\nMejor ({nombre}): AUC={mejor_auc:.4f}  params={mejor_params}")



Tuning: logistic_regression  (20 iteraciones × 4 folds)


  iter 000  AUC=0.9777  Recall=0.9460  params={'clf__C': 3.7554011884736247, 'clf__solver': 'lbfgs'} ◀ mejor


  iter 001  AUC=0.9787  Recall=0.9460  params={'clf__C': 1.844347898661638, 'clf__solver': 'liblinear'} ◀ mejor


  iter 002  AUC=0.9776  Recall=0.9435  params={'clf__C': 5.996584841970366, 'clf__solver': 'lbfgs'}


  iter 003  AUC=0.9777  Recall=0.9460  params={'clf__C': 4.468327528535911, 'clf__solver': 'lbfgs'}


...

  iter 015  AUC=0.9927  Recall=0.9731  params={'clf__estimator__max_depth': 7, 'clf__n_estimators': 28}


  iter 016  AUC=0.9928  Recall=0.9731  params={'clf__estimator__max_depth': 7, 'clf__n_estimators': 18}


  iter 017  AUC=0.9928  Recall=0.9731  params={'clf__estimator__max_depth': 4, 'clf__n_estimators': 13}


  iter 018  AUC=0.9927  Recall=0.9683  params={'clf__estimator__max_depth': 3, 'clf__n_estimators': 23}


  iter 019  AUC=0.9928  Recall=0.9731  params={'clf__estimator__max_depth': 4, 'clf__n_estimators': 18}

Mejor (bagging_pu): AUC=0.9931  params={'clf__estimator__max_depth': 5, 'clf__n_estimators': 19}


## 7. Resumen de resultados

In [8]:
print("\n" + "="*60)
print("RESUMEN HYPERPARAMETER TUNING")
print("="*60)

rows = []
for nombre, res in resultados_tuning.items():
    row = {"modelo": nombre}
    row.update({k: round(v, 4) for k, v in res["best_metrics"].items()})
    row.update({f"param_{k}": v for k, v in res["best_params"].items()})
    rows.append(row)

df_res = pd.DataFrame(rows).set_index("modelo")
cols_metricas = [c for c in df_res.columns if "auc" in c or "recall" in c or "f1" in c]
print(df_res[cols_metricas].to_string())



RESUMEN HYPERPARAMETER TUNING
                     auc_roc_mean  auc_roc_std  f1_mean  f1_std  recall_mean  recall_std
modelo                                                                                  
logistic_regression        0.9789       0.0186   0.7597  0.2076       0.9461      0.0419
random_forest              0.9933       0.0059   0.9737  0.0202       0.9494      0.0382
bagging_pu                 0.9931       0.0059   0.9862  0.0121       0.9731      0.0235


## 8. Guardar mejores hiperparámetros

Se guarda en `data/processed/best_params.json` para que los Prefect flows
del Bloque 4 los carguen en tiempo de ejecución sin hardcodear valores.


In [9]:
best_params_path = _root / "data" / "processed" / "best_params.json"
best_params_path.parent.mkdir(parents=True, exist_ok=True)

params_export = {
    nombre: res["best_params"]
    for nombre, res in resultados_tuning.items()
}

with best_params_path.open("w", encoding="utf-8") as f:
    json.dump(params_export, f, indent=2)

print(f"Guardado: {best_params_path}")
print(json.dumps(params_export, indent=2))


Guardado: C:\Users\Mateo Atehortua\Documents\riesgo-deslizamientos-mlops\data\processed\best_params.json
{
  "logistic_regression": {
    "clf__C": 0.5741157902710026,
    "clf__solver": "liblinear"
  },
  "random_forest": {
    "clf__max_depth": 6,
    "clf__min_samples_leaf": 2,
    "clf__n_estimators": 183
  },
  "bagging_pu": {
    "clf__estimator__max_depth": 5,
    "clf__n_estimators": 19
  }
}


## 9. Guía para actualizar el YAML

Copia los valores encontrados en la sección `modelos` del YAML.
Ejecuta esta celda para ver el bloque YAML listo para copiar-pegar.


## 10. Evaluación en grid completo (métrica primaria — Fix 2)

Los AUC del tuning (CV sobre pseudo-ausencias) siguen siendo ~0.99 por la separabilidad
artificial del dataset. La métrica que reportamos como primaria es el AUC sobre el
**grid completo 2021-2022** (sin filtro de pseudo-ausencias), que refleja el rendimiento
real con el desbalance real (0.40% positivos).

Protocolo de evaluación honesta:
- **Train:** pseudo-ausencias 2019-2020 con mejores hiperparámetros
- **Test:** grid completo 2021-2022 (todas las cuencas × semanas)


In [10]:
from sklearn.base import clone
from experiment.evaluate import evaluar_en_grid_completo

GRID_COMPLETO_PATH = _root / "data" / "processed" / "grid_completo_v3.parquet"
ANIO_TRAIN_FIN = 2020   # holdout estricto: train 2019-2020, eval 2021-2022

# Filtrar datos de entrenamiento a 2019-2020
mask_train = df_model["anio_semana"].apply(lambda p: p.year) <= ANIO_TRAIN_FIN
df_train   = df_model[mask_train]
X_train    = df_train[FEATURES]
y_train    = df_train[TARGET]

print(f"Train 2019-{ANIO_TRAIN_FIN}: {len(df_train)} instancias | pos={y_train.sum()}")
print(f"Eval  2021-2022: grid completo (ver abajo)")
print()

resultados_full = {}

for nombre, res in resultados_tuning.items():
    clf_base = CLASIFICADORES_BASE[nombre]
    pipeline = make_pipeline(clone(clf_base))
    pipeline.set_params(**res["best_params"])

    # Entrenar solo en 2019-2020 (sin contaminar con 2021-2022)
    pipeline.fit(X_train, y_train)

    # Evaluar en grid completo 2021-2022
    metricas_full = evaluar_en_grid_completo(
        pipeline,
        GRID_COMPLETO_PATH,
        feature_cols=FEATURES,
        anio_inicio=2021,
        anio_fin=2022,
    )
    resultados_full[nombre] = metricas_full

    print(f"{nombre}:")
    print(f"  AUC (grid completo 2021-2022) : {metricas_full['auc_roc_full']:.4f}  <- METRICA PRIMARIA")
    print(f"  Recall                        : {metricas_full['recall_full']:.4f}")
    print(f"  Precision                     : {metricas_full['precision_full']:.4f}")
    print(f"  F1                            : {metricas_full['f1_full']:.4f}")
    print(f"  Instancias eval               : {metricas_full['n_total_full']:,} (pos={metricas_full['n_positivos_full']})")
    print()


Train 2019-2020: 4128 instancias | pos=128
Eval  2021-2022: grid completo (ver abajo)



logistic_regression:
  AUC (grid completo 2021-2022) : 0.5686  <- METRICA PRIMARIA
  Recall                        : 0.9401
  Precision                     : 0.0067
  F1                            : 0.0133
  Instancias eval               : 57,096 (pos=334)



random_forest:
  AUC (grid completo 2021-2022) : 0.5817  <- METRICA PRIMARIA
  Recall                        : 0.9701
  Precision                     : 0.0061
  F1                            : 0.0120
  Instancias eval               : 57,096 (pos=334)



bagging_pu:
  AUC (grid completo 2021-2022) : 0.6088  <- METRICA PRIMARIA
  Recall                        : 0.9731
  Precision                     : 0.0060
  F1                            : 0.0120
  Instancias eval               : 57,096 (pos=334)



In [11]:
# Loguear métricas del grid completo en MLflow y actualizar best_params.json

mlflow.set_tracking_uri(cfg.mlflow_tracking_uri)
mlflow.set_experiment(cfg.mlflow.experiment_name)

with mlflow.start_run(run_name="evaluacion_grid_completo_2021_2022"):
    for nombre, mf in resultados_full.items():
        mlflow.log_metrics({
            f"{nombre}__auc_roc_full":    mf["auc_roc_full"],
            f"{nombre}__recall_full":     mf["recall_full"],
            f"{nombre}__precision_full":  mf["precision_full"],
            f"{nombre}__f1_full":         mf["f1_full"],
            f"{nombre}__n_total_full":    mf["n_total_full"],
        })
    mlflow.log_params({
        "train_periodo":   f"2019-{ANIO_TRAIN_FIN}",
        "eval_periodo":    "2021-2022",
        "eval_tipo":       "grid_completo_sin_PA",
        "n_features":      len(FEATURES),
    })

print("Metricas del grid completo logueadas en MLflow.")

# Actualizar best_params.json con métricas del grid completo
best_params_actualizado = {}
for nombre, res in resultados_tuning.items():
    best_params_actualizado[nombre] = {
        **res["best_params"],
        "auc_cv_pa":       round(res["best_metrics"]["auc_roc_mean"], 4),
        "auc_full_grid":   round(resultados_full[nombre]["auc_roc_full"], 4),
        "recall_full":     round(resultados_full[nombre]["recall_full"], 4),
        "precision_full":  round(resultados_full[nombre]["precision_full"], 4),
    }

best_params_path = _root / "data" / "processed" / "best_params.json"
with best_params_path.open("w", encoding="utf-8") as f:
    json.dump(best_params_actualizado, f, indent=2)

print(f"\nbest_params.json actualizado: {best_params_path}")
print(json.dumps(best_params_actualizado, indent=2))


Metricas del grid completo logueadas en MLflow.

best_params.json actualizado: C:\Users\Mateo Atehortua\Documents\riesgo-deslizamientos-mlops\data\processed\best_params.json
{
  "logistic_regression": {
    "clf__C": 0.5741157902710026,
    "clf__solver": "liblinear",
    "auc_cv_pa": 0.9789,
    "auc_full_grid": 0.5686,
    "recall_full": 0.9401,
    "precision_full": 0.0067
  },
  "random_forest": {
    "clf__max_depth": 6,
    "clf__min_samples_leaf": 2,
    "clf__n_estimators": 183,
    "auc_cv_pa": 0.9933,
    "auc_full_grid": 0.5817,
    "recall_full": 0.9701,
    "precision_full": 0.0061
  },
  "bagging_pu": {
    "clf__estimator__max_depth": 5,
    "clf__n_estimators": 19,
    "auc_cv_pa": 0.9931,
    "auc_full_grid": 0.6088,
    "recall_full": 0.9731,
    "precision_full": 0.006
  }
}


In [12]:
# Generar fragmento YAML con los mejores hiperparámetros
lines = ["modelos:"]
for nombre, res in resultados_tuning.items():
    lines.append(f"  {nombre}:")
    for k, v in res["best_params"].items():
        # Convertir clf__param → param (quitar prefijo del pipeline)
        param_name = k.replace("clf__", "").replace("estimator__", "estimator_")
        lines.append(f"    {param_name}: {v}")

print("\n".join(lines))
print("\n# Copia este bloque al final de configs/antioquia_deslizamientos.yaml")


modelos:
  logistic_regression:
    C: 0.5741157902710026
    solver: liblinear
  random_forest:
    max_depth: 6
    min_samples_leaf: 2
    n_estimators: 183
  bagging_pu:
    estimator_max_depth: 5
    n_estimators: 19

# Copia este bloque al final de configs/antioquia_deslizamientos.yaml
